# Advanced Features

This notebook covers advanced features of `seq_tools`, including edit distance calculations, sequence generation, and parallel processing.

## Edit Distance

Edit distance measures how different sequences are from each other. This is useful for:
- Analyzing library diversity
- Quality control
- Understanding sequence similarity


In [ ]:
from seq_tools import (
    calc_edit_distance,
    calc_edit_distance_parallel,
    sequences_to_dataframe,
)

# Create a library of sequences
sequences = [
    "ATCGATCGATCG",
    "ATCGATCGATCA",  # 1 mutation
    "ATCGATCGATCC",  # 1 mutation
    "ATCGATCGATAA",  # 2 mutations
    "GTCGATCGATCG",  # 1 mutation
]
df = sequences_to_dataframe(
    sequences, names=[f"seq_{i+1}" for i in range(len(sequences))]
)

print("Library of sequences:")
print(df[["name", "sequence"]])

# Calculate average edit distance
avg_edit_dist = calc_edit_distance(df)
print(f"\nAverage minimum edit distance: {avg_edit_dist:.2f}")
print("(Lower values indicate more similar sequences)")

### Parallel Edit Distance Calculation

For large libraries, you can use parallel processing to speed up edit distance calculations.


In [ ]:
# For large datasets, use parallel processing
# Note: This is more efficient for libraries with many sequences
import time
from seq_tools import (
    calc_edit_distance,
    calc_edit_distance_parallel,
    sequences_to_dataframe,
)

# Create a larger library
large_library = sequences * 10  # 50 sequences
df_large = sequences_to_dataframe(
    large_library, names=[f"seq_{i+1}" for i in range(len(large_library))]
)

print(f"Calculating edit distance for {len(df_large)} sequences...")

# Standard calculation
start = time.time()
avg_dist_standard = calc_edit_distance(df_large)
time_standard = time.time() - start

# Parallel calculation (4 threads)
start = time.time()
avg_dist_parallel = calc_edit_distance_parallel(df_large, threads=4)
time_parallel = time.time() - start

print(f"Standard method: {avg_dist_standard:.2f} (took {time_standard:.3f}s)")
print(f"Parallel method:  {avg_dist_parallel:.2f} (took {time_parallel:.3f}s)")
print(f"Speedup: {time_standard/time_parallel:.2f}x")

## Sequence Generation

### Generating Mutated Sequences

Generate libraries of mutated sequences from a template.


In [ ]:
from seq_tools import generate_mutated_sequences

# Template sequence
template = "ATCGATCGATCGATCG"

# Generate 10 sequences with 3 mutations each
# You can optionally specify constant 5' and 3' regions
df_mutated = generate_mutated_sequences(
    template=template,
    num_mutations=3,
    num_sequences=10,
    p5_seq="AAAA",  # Constant 5' end
    p3_seq="TTTT",  # Constant 3' end
    ntype="DNA",
)

print(f"Template: {template}")
print(f"Generated {len(df_mutated)} mutated sequences with 3 mutations each:")
print("(Mutations only in the variable region, not in constant ends)")
print(df_mutated[["name", "sequence"]])

### Generating Random Sequences

Create libraries of random sequences with specified nucleotide frequencies.


In [ ]:
from seq_tools import generate_random_sequences

# Generate random sequences
df_random = generate_random_sequences(length=20, num_sequences=10, ntype="DNA")

print("Random DNA sequences:")
print(df_random[["name", "sequence"]])

# Generate random RNA sequences
df_random_rna = generate_random_sequences(length=15, num_sequences=5, ntype="RNA")

print("\nRandom RNA sequences:")
print(df_random_rna[["name", "sequence"]])

## Searching for Patterns in DataFrames

Search for sequence and structural patterns across multiple sequences using DataFrame operations.


In [ ]:
from seq_tools import has_sequence, fold, sequences_to_dataframe

# Create RNA sequences and fold them
rna_seqs = [
    "GGGGUUUUCCCCAAAGGGGUUUUCCCC",  # Two hairpins
    "GCGAAAGC",  # Single hairpin
    "AUGCAUGCAUGC",  # Less structured
]
df_rna = sequences_to_dataframe(
    rna_seqs, names=[f"rna_{i+1}" for i in range(len(rna_seqs))]
)
df_folded = fold(df_rna)

# Search for a sequence pattern
pattern_seq = "GGGGUUUUCCCC"
pattern_struct = "((((....))))"

print("Searching for pattern:")
print(f"  Sequence: {pattern_seq}")
print(f"  Structure: {pattern_struct}")
print("\nResults:")
for idx, row in df_folded.iterrows():
    # Check if sequence contains the pattern
    seq_match = pattern_seq in row["sequence"]
    # Check if structure contains the pattern
    struct_match = pattern_struct in row["structure"]
    both_match = seq_match and struct_match
    print(
        f"  {row['name']}: sequence={seq_match}, structure={struct_match}, both={both_match}"
    )

## Working with CSV Files

The package is designed to work seamlessly with CSV files containing sequences. Here's a workflow example:


In [ ]:
import pandas as pd
import os
from seq_tools import (
    sequences_to_dataframe,
    fold,
    get_molecular_weight_df,
    get_extinction_coeff_df,
    determine_ntype,
    get_length,
)

# Example: Load a CSV file (if you have one)
# df = pd.read_csv("your_sequences.csv")

# For demonstration, create a sample DataFrame
example_df = sequences_to_dataframe(
    ["GGGGUUUUCCCC", "GCGAAAGC", "AUGCAUGCAUGC"], names=["seq1", "seq2", "seq3"]
)

# Typical analysis workflow
print("Starting analysis workflow...")

# 1. Determine nucleotide type
ntype = determine_ntype(example_df)
print(f"1. Nucleotide type: {ntype}")

# 2. Convert to RNA if needed
if ntype == "DNA":
    from seq_tools import to_rna_df

    example_df = to_rna_df(example_df)

# 3. Fold sequences
example_df = fold(example_df)
print("2. Sequences folded")

# 4. Calculate properties
example_df = get_length(example_df)
example_df = get_molecular_weight_df(example_df, ntype="RNA")
example_df = get_extinction_coeff_df(example_df, ntype="RNA")

print("\nFinal DataFrame with all properties:")
print(
    example_df[
        ["name", "sequence", "length", "molecular_weight", "extinction_coeff", "mfe"]
    ]
)

# 5. Save results (optional)
# example_df.to_csv("analyzed_sequences.csv", index=False)

In [ ]:
from seq_tools import (
    validate_sequence,
    validate_dataframe,
    ensure_name_column,
    sequences_to_dataframe,
)
import pandas as pd

# Validate individual sequences
valid_seq = "ATCGATCG"
invalid_seq = "ATCGXYZ"  # Contains invalid characters

print(f"Validating '{valid_seq}': {validate_sequence(valid_seq)}")
print(f"Validating '{invalid_seq}': {validate_sequence(invalid_seq)}")

# Validate DataFrame
df_valid = sequences_to_dataframe(["ATCG", "GCTA"], names=["seq1", "seq2"])
print(f"\nDataFrame is valid: {validate_dataframe(df_valid)}")

# Ensure name column exists
df_no_name = pd.DataFrame({"sequence": ["ATCG", "GCTA"]})
df_with_name = ensure_name_column(df_no_name)
print(f"\nDataFrame with ensured name column:")
print(df_with_name)

## Summary

In this notebook, we've covered:

- ✅ Edit distance calculations for library diversity analysis
- ✅ Parallel processing for large datasets
- ✅ Generating mutated sequence libraries
- ✅ Generating random sequences
- ✅ Pattern matching in DataFrames (sequence and structure)
- ✅ Complete analysis workflows
- ✅ Data validation

## Complete Workflow Example

Here's a complete example combining multiple features:


In [ ]:
# Complete workflow: Generate library, analyze, and evaluate
from seq_tools import (
    generate_mutated_sequences,
    to_rna_df,
    fold,
    get_length,
    get_molecular_weight_df,
    get_extinction_coeff_df,
    calc_edit_distance,
)

# 1. Generate a library of mutated sequences
template = "ATCGATCGATCGATCGATCG"
df_library = generate_mutated_sequences(
    template=template, num_mutations=2, num_sequences=20, ntype="DNA"
)

# 2. Convert to RNA
df_library = to_rna_df(df_library)

# 3. Fold all sequences
df_library = fold(df_library)

# 4. Calculate properties
ntype = "RNA"
df_library = get_length(df_library)
df_library = get_molecular_weight_df(df_library, ntype=ntype)
df_library = get_extinction_coeff_df(df_library, ntype=ntype)

# 5. Calculate library diversity
avg_edit_dist = calc_edit_distance(df_library)

# 6. Summary statistics
print("Library Analysis Summary:")
print(f"  Number of sequences: {len(df_library)}")
print(f"  Average length: {df_library['length'].mean():.1f} nt")
print(f"  Average molecular weight: {df_library['molecular_weight'].mean():.1f} Da")
print(f"  Average MFE: {df_library['mfe'].mean():.2f} kcal/mol")
print(f"  Average edit distance: {avg_edit_dist:.2f}")
print(f"\nTop 5 sequences by MFE (most stable):")
top_5 = df_library.nsmallest(5, "mfe")[["name", "sequence", "structure", "mfe"]]
print(top_5.to_string(index=False))